# 🎙️ All-in-One AI Audio Studio Backend API Engine
### GitHub Dashboard와 실시간 연동되는 고속 배치 오디오 생성 백엔드

이 코랩 노트북은 **백그라운드 GPU 연산 서버** 역할을 수행합니다.
1. 아래의 **[셀 1]**을 실행하면 필수 라이브러리가 설치되고 세션이 1회 자동 재시작됩니다.
2. 재시작 후 **[셀 2]**를 실행하면 FastAPI 백엔드 서버와 Cloudflare 보안 터널이 가동되며 **고유의 HTTPS API 주소**가 발급됩니다.
3. 발급된 주소를 복사하여 **GitHub 대시보드**의 `서버 주소` 입력창에 붙여넣기만 하면 연결이 완료됩니다!

## 📦 [Step 1] 백엔드 필수 패키지 설치 (최초 1회 실행)

In [ ]:
# ① 엔진 및 API 프레임워크 설치
!pip install -q chatterbox-tts "setuptools<81" qwen-tts soundfile fastapi uvicorn python-multipart

# Cloudflare Tunnel (cloudflared) 다운로드
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import os, torch
print("GPU 활성화 상태:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 디바이스:", torch.cuda.get_device_name(0))
else:
    print("⚠️ 상단 메뉴 [런타임] -> [런타임 유형 변경]에서 GPU를 활성화해 주세요!")

FLAG = "/content/.audio_server_setup_done"
if not os.path.exists(FLAG):
    open(FLAG, "w").close()
    print("\n[1단계 완료] 세션을 자동 재시작합니다. 잠시 후 [Step 2]를 실행하세요!")
    os.kill(os.getpid(), 9)
else:
    print("\n환경 준비 완료! 아래 [Step 2]를 실행하세요.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 97.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


## ⚡ [Step 2] 모델 로드 & 백엔드 API 서버 가동
실행 즉시 고유의 `https://xxxx.trycloudflare.com` 주소가 하단에 출력됩니다.

In [ ]:
import os, sys, time, subprocess, threading, re
import torch, torchaudio, soundfile as sf
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response, JSONResponse
import uvicorn

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 모델 초기화 중... (Device: {DEVICE})")

# 1. Chatterbox Multilingual 로드
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
cb_model = ChatterboxMultilingualTTS.from_pretrained(device=DEVICE)
print("✅ Chatterbox Multilingual TTS 준비 완료")

# 2. Qwen3-TTS 로드
from qwen_tts import Qwen3TTSModel
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
qwen_model = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-0.6B-Base", device_map="cuda:0" if torch.cuda.is_available() else "cpu", dtype=dtype)
print("✅ Qwen3-TTS 준비 완료")

# 3. FastAPI 인스턴스 생성 및 CORS 허용
app = FastAPI(title="AI Audio Studio API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

STORAGE_DIR = "/content/studio_cache"
os.makedirs(STORAGE_DIR, exist_ok=True)

@app.get("/health")
def health_check():
    return {"status": "ok", "device": DEVICE, "models": ["chatterbox", "qwen3"]}

@app.post("/api/synthesize")
async def synthesize(
    voice_file: UploadFile = File(...),
    text: str = Form(...),
    engine: str = Form("chatterbox"), # "chatterbox" 또는 "qwen3"
    lang: str = Form("ko"),           # en, ko, ja, es, fr, zh, etc.
    ref_text: str = Form(""),         # Qwen3용 참조 발화문
    cfg_weight: float = Form(0.5),
    exaggeration: float = Form(0.5)
):
    try:
        # 목소리 파일 저장 및 24kHz 변환
        raw_path = f"{STORAGE_DIR}/temp_raw_{int(time.time()*1000)}.wav"
        with open(raw_path, "wb") as f:
            f.write(await voice_file.read())

        clean_voice = f"{STORAGE_DIR}/clean_{int(time.time()*1000)}.wav"
        subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", raw_path, "-ac", "1", "-ar", "24000", "-t", "30", clean_voice], check=True)

        out_wav_path = f"{STORAGE_DIR}/out_{int(time.time()*1000)}.wav"

        if engine == "qwen3":
            q_ref = ref_text.strip() if ref_text.strip() else "안녕하세요. 인공지능 오디오 스튜디오 목소리 샘플입니다."
            wavs, sr = qwen_model.generate_voice_clone(
                text=text,
                language="Korean" if lang in ["ko", "Korean"] else "English",
                ref_audio=clean_voice,
                ref_text=q_ref
            )
            sf.write(out_wav_path, wavs[0], sr)
        else:
            # Chatterbox Multilingual
            wav = cb_model.generate(
                text,
                language_id=lang,
                audio_prompt_path=clean_voice,
                exaggeration=exaggeration,
                cfg_weight=cfg_weight
            )
            torchaudio.save(out_wav_path, wav.cpu(), cb_model.sr)

        with open(out_wav_path, "rb") as f:
            wav_bytes = f.read()

        # 임시 파일 정리
        for p in [raw_path, clean_voice, out_wav_path]:
            if os.path.exists(p): os.remove(p)

        return Response(content=wav_bytes, media_type="audio/wav")
    except Exception as e:
        print("생성 오류 발생:", str(e))
        raise HTTPException(status_code=500, detail=str(e))

# Uvicorn 백그라운드 구동
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

# Cloudflare Tunnel 기동
print("🌐 공개 HTTPS 터널 생성 중...")
tunnel_cmd = ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"]
tunnel_proc = subprocess.Popen(tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print("\n" + "="*60)
print("🎉 [백엔드 API 서버 가동 완료]")
print("아래 주소를 복사하여 깃허브 대시보드 [서버 주소]에 입력하세요:")
print(f"👉 {public_url}")
print("="*60 + "\n")

# 프로세스 유지
tunnel_proc.wait()
